# Lab - Unit 10: model predictive control on the real car

**Runs on your machine with CARLA 0.9.12 and `ROAR_PY`.**

On racekit, Unit 10 replaced pure pursuit's one look-ahead point with a plan of the whole next 2 s, made again every tick: a linear time-varying model of the errors from the line, one least-squares solve per tick, the first command applied.
The model knows the measured lag of 1 tick (Unit 8) and the measured steering gain (66 deg); it is a kinematic model, and the car is not, so its weights were tuned on racekit's dynamic carla-like car, on the training track only.
At the same `MU_FRACTION` it was no faster than pursuit; what it bought was **tracking margin**, which let the fraction rise from 0.54 to 0.58 on the training family, and on the unseen `test-circuit-seed2026` it raced 276.85 s where Unit 9 raced 282.05 s.
**In this lab you find out whether that margin exists on CARLA's car**, starting from the configuration your earlier labs accepted and racing a short, fixed procedure that never reports a setting it did not race.

What you do, in order:

1. name your starting point: the **accepted configuration** (§2), or race for one;
2. optionally add a Unit 9 line to it (R0a-R0b);
3. check the MPC file's timing on your own machine, off CARLA (step 1);
4. race the MPC file twice (R1-R2), and, if it is accepted, one fraction step higher twice (R3-R4);
5. never commit your results.

How to use it:

- The cells tagged `no-exec` are for **your machine** (the Windows PC you set up in the Unit 1 lab).
  Their shell lines are shown with `!` only so the notebook stays valid.
- The analysis cells run anywhere.
  In CI they run against a racekit-made sample recording (the starter on `monza-like`, with no respawn), so the book always proves they work; on your machine they run against your own recordings, **one at a time**.

> **Never commit your results.**
> This repository is public.
> Your recordings (the telemetry CSVs and any `.sections.txt` sidecars), screenshots and logs stay on your own machine, **outside the repository**
> (the Unit 1 lab uses `%USERPROFILE%\roar-runs\`).
> The repository ignores data files under `labs/`, and a CI test fails if anything other than a notebook
> or a README is ever tracked there, but the first line of defence is not putting them here at all.

In [ ]:
import os

TELEMETRY_PATH = os.environ.get("ROAR_LAB_CSV", "telemetry.csv")

The parameters cell above is the **only** place a recording's file name appears.
Set `ROAR_LAB_CSV` to one recording's full path (outside the repository) before you start Jupyter; the recorder writes to the same variable.
Every race of this lab is its own recording, so give each its own name (the commands in §3 do) and point `ROAR_LAB_CSV` at the one you are analysing.

## 1. The five steps

The Unit 1 lab (`units/unit-01-the-race-and-baseline/labs/real-sim.ipynb`) has every detail; this lab only changes what you race.

1. **Start the Monza v1.1 executable** (CARLA 0.9.12) and wait for the map to load (Unit 1 lab, §1 for the pinned versions and §3 for the start).
2. **Put the submission in the forked competition folder**: the file §3's table names for the run, with the Unit 1 recorder and its three wiring lines, Unit 5's one-line `offset` change and Unit 7's section sidecar pasted in, exactly as for your earlier labs.
   `agents/unit-10.py` has the same `initialize()`, `advance_index` line and `measure()` as `agents/unit-05.py`, so the pasted parts go in the same places.
   Before you go to CARLA, race the edited file on racekit and run `check_submission` on it, as the Unit 1 lab says.
   A file whose `LINE_CONTROL` holds a line solved for CARLA's Monza waypoints cannot race on racekit's `test-circuit` (the agent refuses a line solved for another track): run those two checks on a copy with `LINE_CONTROL = None` and nothing else changed (step 1 below says why that is fair).
3. **Run `competition_runner.py`** from `ROAR_Competition\competition_code`, with `ROAR_LAB_CSV` set to a new file outside the repository.
   Note the time and the collision count the runner reports, and whether the race finished.
4. **Export the telemetry to CSV**: the recorder does it as the car drives, one row per tick, to `ROAR_LAB_CSV`.
5. **Analyse it**: start Jupyter with `ROAR_LAB_CSV` pointing at the recording and run §4 onwards of this notebook.

**The `offset` column is the error this lab judges.**
Since Unit 5's change the recorder writes the agent's own cross-track error, `self.cross_track_error`, and Units 9 and 10 measure it from their **own path**: the racing line when `LINE_CONTROL` is set, the centreline when it is `None`.
So on CARLA the column is already the line-relative error of the clean rule, with nothing to subtract.

## 2. The starting point: your accepted configuration

The **accepted configuration** is the exact submission your **Unit 8 lab accepted**: a setting that was clean on both of its CARLA races.
It is either the Unit 8 file with your `MU_G`, `MU_FRACTION` and `MAX_STEER` (the grip identified), or your Unit 7 submission with `MAX_STEER` changed to your gain (the no-estimate path, which plans with `A_LAT` and has no `MU_FRACTION`).
It has been raced clean on CARLA, so it is the only safe fallback: whatever happens below, you can always go back to it, and nothing that has not raced clean ever takes its place.
Write down its two race times: they are its **record**, the times MPC has to be compared with.

Two other starting points:

- **Your Unit 8 lab accepted no setting**, and you hold only a **PROVISIONAL** Unit 7 result (the Unit 7 lab's step 4: all knobs at 1.0 at your `A_LAT`, not clean on both re-races).
  A provisional result has not raced clean twice, so it is not accepted yet: **R00a-R00b** race that Unit 7 submission exactly as it is, twice.
  Clean on both: it becomes the accepted configuration (its two times are its record), and the lab goes on.
  Otherwise the lab records **"no accepted configuration"** and stops: MPC is never compared against a configuration that has not raced clean.
- **You have no clean CARLA race at all** (the Unit 7 lab not run, or nothing clean in it): the lab records "no accepted configuration" and stops before any race.

**R0a-R0b (optional): add a Unit 9 line.**
Only if you have solved a line for CARLA's Monza waypoints with Unit 9's solver.
Race the accepted configuration with your line and pure pursuit, twice: your Unit 9 file, `agents/unit-09.py`, with the accepted configuration's constants and your `LINE_CONTROL`.
Clean on both: the line joins the accepted configuration (and its two times become the record).
Otherwise the line is dropped and the accepted configuration is unchanged.

## 3. The runs

| run | file | constants changed at the top of the file |
|---|---|---|
| **R00a, R00b** (only from a PROVISIONAL Unit 7 result) | your Unit 7 lab file | none: exactly as the Unit 7 lab raced it |
| **R0a, R0b** (optional) | `agents/unit-09.py` with the pasted parts | the accepted configuration's constants, and `LINE_CONTROL` = your line |
| **step 1** (off CARLA) | your Unit 10 file | as for R1 |
| **R1, R2** | `agents/unit-10.py` with the pasted parts: **your Unit 10 file** | the accepted configuration's constants copied in: `LINE_CONTROL` (your line, or `None`), `MU_G` (or `None`, with your `A_LAT` and `GRIP`), `MU_FRACTION`, `MAX_STEER`; and `MPC_HORIZON` = 20 if step 1 needed it |
| **R3, R4** (only after MPC is accepted, and only when the accepted configuration plans with `MU_FRACTION`) | your Unit 10 file | as for R1-R2, with `MU_FRACTION` one step (0.02) higher |

Everything else in the Unit 10 file stays as shipped: `STEERING = "mpc"`, the weights `MPC_Q_E = 0.3`, `MPC_Q_PSI = 20.0`, `MPC_R_DU = 1000.0`, `MPC_LAG = 1` and the tick budget `MPC_BUDGET_MS = 5.0`.

**The weights are un-probed off racekit's line.**
They were tuned at 0.58 on racekit's racing line; nobody has raced them on a centreline (`LINE_CONTROL = None`) or on the `A_LAT` path.
R1-R2 gate the MPC file either way, which is what they are for: if those weights do not suit your configuration, R1-R2 say so, and you keep the accepted configuration.

**The tick budget is live on CARLA.**
On racekit every pinned race switched the timing off (`MPC_BUDGET_MS = None`) so that it never depended on the machine.
On CARLA the shipped budget is on: an `mpc()` call slower than 5 ms halves the horizon for the rest of the race (40, 20, then 10 ticks), and the agent prints one line each time it does, in the runner's console.
**Write those lines down by hand** for every MPC race; the analysis below does not read them.

In [ ]:
# Anaconda Prompt, on the Windows machine, from ROAR_Competition\competition_code. Not run in CI.
# Type each line WITHOUT the leading "!" (it is there only so the notebook stays valid).
# 1. Your Unit 10 file: the shipped agent, then the recorder, Unit 5's change and the sidecar pasted in,
#    and the accepted configuration's constants copied in (section 3's table)
!copy %PY4KIDS%\roar\projects\racekit\agents\unit-10.py submission.py
# 2. Print the SHA-256 of the file you are about to race, and write it down
!certutil -hashfile submission.py SHA256
# 3. One recording per run, each with its own name (R00a, R00b, R0a and R0b race their own files, section 3)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-10-r1.csv" && python competition_runner.py
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-10-r2.csv" && python competition_runner.py
# ... and so on: unit-10-r00a, unit-10-r00b, unit-10-r0a, unit-10-r0b, unit-10-r3, unit-10-r4, as section 5 asks
# 4. Analyse one recording: start Jupyter with the variable pointing at it (from your clone of this repository)
!set "ROAR_LAB_CSV=%USERPROFILE%\roar-runs\unit-10-r1.csv" && jupyter lab

**Step 1: the timing, first, off CARLA.**
Before any MPC race, run `check_submission` on **exactly the file R1 will race** (your Unit 10 file with the accepted constants), on the machine that will run the agent, with nothing else running.
It races the file on racekit and measures `step()`: the p99 after 50 warm-up steps must be at most 5 ms.
Never judge timing while other races or programs share the machine: a busy machine slows every tick, and the p99 then measures the machine, not the agent.

**A Monza line in the file.** `check_submission` races the file on racekit's `test-circuit`, and the agent refuses a `LINE_CONTROL` solved for another track (it raises `ValueError`, so the report says FAIL without measuring any timing).
If your R1 file carries a Monza line, run the check on a **timing copy**: the same file with `LINE_CONTROL = None` and nothing else changed.
The line only moves the waypoints; the horizon, the matrix sizes and so the cost of each solve are the same, so the copy's p99 is the R1 file's.
Then the R1 file is the copy with your line put back; the line itself was already raced clean on CARLA in R0a-R0b.
Write in §5 which file the check timed.

- It passes: `CHECK_40 = True` in §5.
- It fails: set `MPC_HORIZON = 20` in the file, run the check again, and log the result as `CHECK_20`.
  A shorter horizon is cheaper and, on racekit, barely slower: the plan's probes raced 20 ticks clean on every family and unseen-track race at 0.58.
  R1-R2 then race the file with 20.
- It fails at 20 too: the lab keeps the accepted configuration and stops. Your machine is too slow for this MPC, and that is a result.

In [ ]:
# On your machine, in this notebook, with the CARLA server NOT running. Not run in CI.
from pathlib import Path

from racekit.check_submission import check_submission

MPC_FILE = Path(r"C:\path\to\ROAR_Competition\competition_code\submission.py")  # your Unit 10 file
# With a Monza LINE_CONTROL in it, point MPC_FILE at the timing copy (LINE_CONTROL = None) instead.
report = check_submission(MPC_FILE)
print(report)
print("CHECK passed:", report.ok)

## 4. Analyse one recording

Set `RACE_FINISHED` and `RUNNER_COLLISIONS` to what the runner reported (Unit 1 lab, §4, explains why the recorder cannot know).
The cell applies the unit's **clean rule** to the recording: the race finished, with 0 collisions and no respawn, and the max |cross-track error| from the agent's own path (the `offset` column, §1) is at most 2 m over the driving rows (`held == 0`), without the stale first row after a hold.
If the race is complete, it also prints the race time, from the sector splits as the Unit 1 lab computes it.
Log the run in §5 with that time.

In [ ]:
from math import isnan

import matplotlib.pyplot as plt
import numpy as np
from racekit.telemetry import read_csv, sector_splits

RACE_FINISHED = True  # set False if the runner reported that the race did not finish
RUNNER_COLLISIONS = 0  # the collision count the runner reported
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the max |cross-track error| from the agent's own path


def driving_rows(telemetry):
    # The rows the clean rule judges: not held, and not the stale first row after a hold run.
    held = telemetry["held"] == 1
    keep = ~held
    stale = [row + 1 for row in np.nonzero(held)[0] if row + 1 < len(held) and not held[row + 1]]
    keep[np.array(stale, dtype=int)] = False
    return keep


def clean_check(telemetry):
    # (clean, respawns, max |error| in m) by the unit's rule
    held = telemetry["held"] == 1
    respawns = len([row for row in np.nonzero(held)[0] if row == 0 or not held[row - 1]])
    keep = driving_rows(telemetry)
    worst = float(np.max(np.abs(telemetry["offset"][keep]))) if np.sum(keep) else float("nan")
    is_clean = RACE_FINISHED and RUNNER_COLLISIONS == 0 and respawns == 0 and worst <= CLEAN_ERROR
    return is_clean, respawns, worst


def race_time(telemetry, meta):
    # The race time from the sector splits (Unit 1 lab), or None when a sector was never reached.
    splits = sector_splits(telemetry, meta["progress_total"], laps=meta["laps"], per_lap=meta["per_lap"],
                           finished=RACE_FINISHED)
    if any(isnan(split) for lap in splits for split in lap):
        return None
    held_rows = np.nonzero(telemetry["held"] == 1)[0]
    restart = float(telemetry["t"][held_rows[-1]]) if len(held_rows) else 0.0  # after the last hold
    return restart + float(sum(sum(lap) for lap in splits))


telemetry, meta = read_csv(TELEMETRY_PATH)
is_clean, respawns, worst = clean_check(telemetry)
this_time = race_time(telemetry, meta)
print(f"your recording: {len(telemetry)} rows, {respawns} respawns, "
      f"max |cross-track error from its own path| {worst:.3f} m")
print(f"clean (finished, 0 collisions, no respawn, max |e| <= {CLEAN_ERROR} m): {is_clean}")
print("race time: " + ("not complete" if this_time is None else f"{this_time:.2f} s"))
if worst == 0.0:
    print("every offset is 0: was this recorded without Unit 5's one-line change?")

In [ ]:
keep = driving_rows(telemetry)
fig, (ax_error, ax_speed) = plt.subplots(1, 2, figsize=(12, 3.5))
ax_error.plot(telemetry["t"][keep], telemetry["offset"][keep], linewidth=0.6)
for limit in (-CLEAN_ERROR, CLEAN_ERROR):
    ax_error.axhline(limit, color="grey", linestyle="--", linewidth=1.0)
ax_error.set_xlabel("t (s)")
ax_error.set_ylabel("cross-track error (m)")
ax_error.set_title("the error from the agent's own path, and the 2 m rule (dashed)")
ax_speed.plot(telemetry["t"], telemetry["v"], linewidth=0.6)
ax_speed.set_xlabel("t (s)")
ax_speed.set_ylabel("speed (m/s)")
ax_speed.set_title("speed")
fig.tight_layout()
plt.show()

**What racekit predicts** (the carla-like car, plan `roar-009`; the milestone `milestones/unit-10-mpc.ipynb` asserts every number):

| racekit, `carla_like(noise=True)` | pure pursuit (Unit 9) | MPC (Unit 10) |
|---|---|---|
| `test-circuit`, same fraction 0.54 | 279.60 s | 279.65 s (seed 0): neither noise-aware better |
| `test-circuit-seed4`, 0.58 | not clean | 271.10 s, clean |
| `test-circuit-seed2026` (unseen) | 282.05 s at 0.54 | 276.85 s at 0.58 |
| `monza-like`, 0.64 | finishes, but just over the 2 m rule | 468.75 s, clean |

racekit is not CARLA, and `monza-like` is not CARLA's Monza: compare the **pattern** (does MPC stay further inside the 2 m rule than your accepted configuration did, at the same fraction?), not the second decimal.
At the same fraction, expect MPC to be **no faster**: the speed profile sets the lap time.
Its gain, if it has one on CARLA, shows up at R3-R4, one fraction step higher.

## 5. The decision procedure

1. **The starting point** (§2): `START = "accepted"` (your Unit 8 lab accepted a setting), `"provisional"` (it accepted none and you hold a PROVISIONAL Unit 7 result: R00a-R00b first), or `"none"` (stop).
   `ACCEPTED_FRACTION` is its `MU_FRACTION`, or `None` on the `A_LAT` path; `ACCEPTED_TIMES` are its record.
2. **R0a-R0b** only when `TRY_LINE = True`.
3. **Step 1**, `CHECK_40` and, if it failed, `CHECK_20`.
4. **R1-R2**: the MPC file, twice, stopping at the first race that is not clean.
   Both clean: **MPC is accepted at the accepted configuration's fraction**.
   Otherwise you keep the accepted configuration, and there are no further races.
5. **R3-R4**, only after MPC is accepted and only when the accepted configuration plans with `MU_FRACTION`: the fraction + 0.02, twice.
   Both clean: MPC is accepted at the higher fraction; otherwise the accepted fraction stays the one R1-R2 raced.

The procedure **never ends on a value it did not race**: its result is always a configuration that was clean on both of its races, or "no accepted configuration".

**The run budget**: at most **8 races** (R00a-R00b, R0a-R0b, R1-R4), about 80 minutes at about 10 minutes a race.
In practice it is at most 6: a PROVISIONAL Unit 7 result is on the `A_LAT` path, which skips R3-R4.
The **short form** is R1-R2 only: **2 races**, from an accepted configuration, with no line and no higher fraction.
A host failure (the server exits, or the agent gets no first tick) is repeated and not counted: it says nothing about the agent.

Log every counted race in `RUNS`, in the order you raced them, as `(name, clean by §4's rule, race time in s or None)`, and run the cell: it replays the procedure on your log, checks that each race is the one the procedure asked for (and that none is logged after the procedure ended), and prints the next step, the result, and the result's median time against the record of the accepted configuration MPC would replace.

In [ ]:
from racekit.measure import noise_aware_better

FRACTION_STEP = 0.02  # R3-R4: one step above the accepted fraction
RACES_PER_SETTING = 2  # a setting is accepted only when it is clean on both of its races
RUN_BUDGET = 8  # R00a-R00b, R0a-R0b, R1-R4

START = "accepted"  # "accepted", "provisional" or "none" (section 2)
ACCEPTED_FRACTION = 0.54  # the accepted configuration's MU_FRACTION, or None on the A_LAT path
ACCEPTED_TIMES = []  # its record: the two clean race times (s) from the lab that accepted it
TRY_LINE = False  # True: R0a-R0b add your Unit 9 line first
SHORT_FORM = False  # True: R1-R2 only, from an accepted configuration
CHECK_40 = None  # step 1: True / False once check_submission has run on the MPC file, None before
CHECK_20 = None  # step 1 again with MPC_HORIZON = 20, only if CHECK_40 is False
# Your races, in the order you raced them: (name, clean by section 4's rule, race time in s or None),
# with the names R00a, R00b, R0a, R0b, R1, R2, R3, R4.
RUNS = []


class NextRun(Exception):
    """The procedure needs a step that is not in the log yet."""


def replay(start, accepted_fraction, accepted_times, runs, check_40, check_20, try_line=False, short_form=False):
    """Replay the procedure: returns (the next step or None, the notes, the result or None, the record of the
    accepted configuration MPC is compared with, the result's times)."""
    notes, log = [], list(runs)
    record = {"label": "the accepted configuration", "times": list(accepted_times)}
    first_record = dict(record)

    def race(name, what):
        if not log:
            raise NextRun(f"{name}: {what}")
        logged, clean, seconds = log.pop(0)
        if logged != name:
            raise ValueError(f"race {len(runs) - len(log)} is {logged}, but the procedure asked for {name} ({what})")
        shown = "" if seconds is None else f", {seconds:.2f} s"
        notes.append(f"race {len(runs) - len(log)}, {name}, {what}: {'clean' if clean else 'NOT clean'}{shown}")
        return clean, seconds

    def both(names, what):
        # race the setting twice, stopping at the first race that is not clean; (both clean, their times)
        times = []
        for name in names:
            clean, seconds = race(name, what)
            if not clean:
                return False, times
            times.append(seconds)
        return True, times

    def finish(result, times):
        if log:
            raise ValueError(f"{len(log)} race(s) logged after the procedure ended; remove them or fix the log")
        notes.append(f"the result: {result}")
        return None, notes, result, first_record, times

    try:
        if short_form and (start != "accepted" or try_line):
            raise ValueError("the short form is R1-R2 only, from an accepted configuration, without a line")
        if start == "none":
            return finish("no accepted configuration (no clean CARLA race): stop", [])
        fraction = accepted_fraction
        if start == "provisional":
            fraction = None  # a Unit 7 submission plans with A_LAT
            accepted, times = both(("R00a", "R00b"), "your Unit 7 submission, exactly as it is")
            if not accepted:
                return finish("no accepted configuration (the Unit 7 submission was not clean on both R00 races): "
                              "stop", [])
            record = {"label": "your Unit 7 submission (R00a-R00b)", "times": times}
            first_record = dict(record)
            notes.append("the Unit 7 submission is the accepted configuration")
        elif start != "accepted":
            raise ValueError(f"START must be 'accepted', 'provisional' or 'none', not {start!r}")
        plans = "MU_FRACTION " + str(fraction) if fraction is not None else "A_LAT"
        line = ""
        if try_line:
            accepted, times = both(("R0a", "R0b"), "the accepted configuration with your line, pure pursuit")
            if accepted:
                line = ", with your line"
                record = {"label": "the accepted configuration with your line (R0a-R0b)", "times": times}
                notes.append("the line joins the accepted configuration")
            else:
                notes.append("the line is dropped; the accepted configuration is unchanged")
        first_record = dict(record)  # what MPC is compared with: the accepted configuration it would replace
        keep = f"keep {record['label']} ({plans}{line})"
        if check_40 is None:
            raise NextRun("step 1: check_submission on the MPC file (MPC_HORIZON = 40), on the idle machine")
        horizon = 40
        if not check_40:
            if check_20 is None:
                raise NextRun("step 1: MPC_HORIZON = 20 in the MPC file, then check_submission again")
            if not check_20:
                return finish(f"{keep}: the MPC file fails check_submission at a horizon of 20 too; stop",
                              record["times"])
            horizon = 20
        elif check_20 is not None:
            raise ValueError("CHECK_20 is only for an MPC file that failed at a horizon of 40")
        mpc = f"the MPC file, {plans}, horizon {horizon}{line}"
        accepted, times = both(("R1", "R2"), mpc)
        if not accepted:
            return finish(f"{keep}: MPC was not clean on both R1 and R2", record["times"])
        if short_form or fraction is None:
            if fraction is None:
                notes.append("the accepted configuration plans with A_LAT: R3-R4 are skipped")
            return finish(f"MPC accepted: {mpc}, clean on R1 and R2", times)
        higher = round(fraction + FRACTION_STEP, 2)
        accepted, higher_times = both(("R3", "R4"), f"the MPC file, MU_FRACTION {higher}, horizon {horizon}{line}")
        if accepted:
            return finish(f"MPC accepted at MU_FRACTION {higher}, horizon {horizon}{line}, clean on R3 and R4",
                          higher_times)
        return finish(f"MPC accepted: {mpc}, clean on R1 and R2 ({higher} was not clean on both R3 and R4)", times)
    except NextRun as needed:
        return str(needed), notes, None, first_record, []


# Worked examples (made up, not recordings).
# Accepted at 0.54, the check passes, all four races clean: MPC at 0.56, after 4 races.
four = [("R1", True, 330.1), ("R2", True, 329.8), ("R3", True, 327.9), ("R4", True, 328.4)]
step, notes, result, start_record, times = replay("accepted", 0.54, [331.0, 330.6], four, True, None)
assert step is None and result.startswith("MPC accepted at MU_FRACTION 0.56") and times == [327.9, 328.4]
# R2 not clean: keep the accepted configuration, and a race logged after that is rejected.
two = [("R1", True, 330.1), ("R2", False, None)]
step, notes, result, _, times = replay("accepted", 0.54, [331.0, 330.6], two, True, None)
assert step is None and result.startswith("keep the accepted configuration") and times == [331.0, 330.6]
try:
    replay("accepted", 0.54, [331.0, 330.6], two + [("R3", True, 327.0)], True, None)
except ValueError as extra:
    assert "after the procedure ended" in str(extra)
else:
    raise AssertionError("an extra race was accepted")
# R3 not clean: MPC stays at the fraction R1-R2 raced, never at the un-raced 0.56.
step, notes, result, _, times = replay("accepted", 0.54, [], four[:2] + [("R3", False, None)], True, None)
assert result.startswith("MPC accepted: the MPC file, MU_FRACTION 0.54") and times == [330.1, 329.8]
# A PROVISIONAL Unit 7 result that is not clean on R00a: no accepted configuration, and no MPC race.
step, notes, result, _, _ = replay("provisional", None, [], [("R00a", False, None)], None, None)
assert step is None and result.startswith("no accepted configuration")
# A PROVISIONAL result clean twice, a line, a slow machine (20 ticks), MPC clean: A_LAT, so no R3-R4 (6 races).
six = [("R00a", True, 335.2), ("R00b", True, 334.9), ("R0a", True, 333.0), ("R0b", True, 333.4),
       ("R1", True, 332.5), ("R2", True, 332.1)]
step, notes, result, start_record, times = replay("provisional", None, [], six, False, True, try_line=True)
assert step is None and result == "MPC accepted: the MPC file, A_LAT, horizon 20, with your line, clean on R1 and R2"
assert start_record["times"] == [333.0, 333.4] and len(six) <= RUN_BUDGET  # the record is now the line's
# The check fails at 40 and at 20: keep the accepted configuration, no MPC race; no clean race at all: stop.
assert replay("accepted", 0.54, [], [], False, False)[2].startswith("keep the accepted configuration")
assert replay("none", None, [], [], None, None)[2].startswith("no accepted configuration")
# An unfinished log names the next step.
assert replay("accepted", 0.54, [], [], None, None)[0].startswith("step 1: check_submission")
assert replay("accepted", 0.54, [], four[:1], True, None)[0].startswith("R2: the MPC file, MU_FRACTION 0.54")

next_step, notes, result, start_record, result_times = replay(START, ACCEPTED_FRACTION, ACCEPTED_TIMES, RUNS,
                                                              CHECK_40, CHECK_20, TRY_LINE, SHORT_FORM)
print("\n".join(notes) if notes else "no races logged yet")
print(f"races: {len(RUNS)} logged, at most {2 if SHORT_FORM else RUN_BUDGET}")
print(f"next: {next_step}" if next_step else "the procedure is complete")
if result is not None and len(start_record["times"]) == 2 and len(result_times) == 2 and None not in result_times:
    print(f"the result's median {np.median(result_times):.2f} s against {start_record['label']}'s "
          f"{np.median(start_record['times']):.2f} s; noise-aware better: "
          f"{noise_aware_better(result_times, start_record['times'])} (two races each: weak evidence)")

## What to take away

- A controller is accepted on the real car the way a fraction is: clean on **both** of its races, against a configuration that has itself raced clean.
- MPC's promise is **margin**, not speed at the same fraction: look for a smaller max error first, and a higher fraction second.
- Timing is measured on the machine that will run the agent, idle, before the race; on the race, a slow tick shortens the horizon and the agent says so.
- The model is kinematic and the car is not: weights tuned on racekit are a starting point, and R1-R2 are what decide.

> **One last check: never commit your results.**
> This repository is public.
> Your recordings, screenshots and logs stay on your own machine, in the folder outside the repository
> that `ROAR_LAB_CSV` pointed at.
> Run `git status` in your clone before you commit anything: none of them should be listed.